# 🔧 CPI Bangladesh Mission — Phase 2 Fixes & Enhancements
### Continuation from DriveBuilder v1.0

**Run after:** `000_BGD_CPMS_DriveBuilder_v1.0.ipynb`  
**Steps:** 15 (Fix naming) · 16 (Upload exports) · 17 (Populate INDEX) · 18 (Permissions)

---

In [ ]:
# ── Re-run auth + helpers (must run before any other cell) ──────
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import json, time, io
from googleapiclient.http import MediaIoBaseUpload

drive_service  = build('drive',  'v3')
sheets_service = build('sheets', 'v4')

SHARED_DRIVE_ID = '18N_9Eq4oOCPZoTBW4CbsVNqLVBkvb6KW'
TODAY = '2026-04'
VER   = 'v1.0'

# Re-declare helper functions
def find_item(name, parent_id, mime_type=None):
    q = f"name='{name}' and '{parent_id}' in parents and trashed=false"
    if mime_type:
        q += f" and mimeType='{mime_type}'"
    try:
        results = drive_service.files().list(
            q=q, spaces='drive', fields='files(id, name)',
            supportsAllDrives=True, includeItemsFromAllDrives=True
        ).execute()
        files = results.get('files', [])
        return files[0]['id'] if files else None
    except HttpError:
        return None

def create_folder(name, parent_id):
    existing = find_item(name, parent_id, 'application/vnd.google-apps.folder')
    if existing:
        print(f'  📁 [EXISTS]  {name}')
        return existing
    meta = {'name': name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [parent_id]}
    f = drive_service.files().create(body=meta, fields='id', supportsAllDrives=True).execute()
    print(f'  📁 [CREATED] {name}')
    time.sleep(0.15)
    return f['id']

def section(t):
    print(f"\n{'='*60}\n  {t}\n{'='*60}")

# Restore live folder IDs from the v1.0 run
ids = {
    '00_TPL':    '1Qkyi-ZdzvG_52-aFp471nzAvTCLMUAzG',
    '01_HRA':    '18iryBvBOZr96FR49zZoVO6q5QZQX3wdp',
    '02_GF':     '1zDhsKUZEQi50mQjpdU8TcvmaMPJ2Tf-N',
    '03_LSC':    '1U_Tf3Z-kYAurOTkIZ-BLhrqy3dIDYaHo',
    '04_MER':    '1LH5oT8sIb0A_taNNok2HBAZ0C6PWE4Qj',
    '01_GOV':    '1G-Gv9WDFuEwg8sogFYf8jjag_USjiipk',
    '02_FIN':    '1mNZ18VrLZGVhBdD8_wuuwegJyjXHt4zV',
    '03_HRA_G':  '1DSQW6C8vV1fuBAqNldmnprKU4x3m2XZQ',
    '04_LOG':    '1rwVQDKpGQeSWQNvOKNSWPb5K7M_98vuA',
    '05_PROG_G': '1408RoCljObPWIVhGkl58nntK0-9Dd_ex',
    '06_MER_G':  '1MCzNI8dQTNsoS4fWXg1ib-bKIflOzpW_',
    '07_PART':   '1-WaJfRHzcozL58xFdYGy8zA2jJLPhMc6',
    '08_COMM':   '1WTi2Sqtzz45DUvGJV9uWXggZv317wazk',
}
ROOT = SHARED_DRIVE_ID
print('✅ Setup complete. IDs restored.')

## STEP 15 — Fix Global Folder Naming Conflict

Renames the 8 Step-11 Global Governance folders to use `GBL_` prefix so they sort separately from Bangladesh Mission folders.

In [ ]:
section('STEP 15 — Rename Global Governance folders → GBL_ prefix')

# Old name → New name mapping
renames = {
    '01_GOV':    ('01_Governance',     'GBL_01_Governance'),
    '02_FIN':    ('02_Finance',        'GBL_02_Finance'),
    '03_HRA_G':  ('03_HR_Admin',       'GBL_03_HR_Admin'),
    '04_LOG':    ('04_Logistics',      'GBL_04_Logistics'),
    '05_PROG_G': ('05_Programs',       'GBL_05_Programs'),
    '06_MER_G':  ('06_MER',            'GBL_06_MER'),
    '07_PART':   ('07_Partnerships',   'GBL_07_Partnerships'),
    '08_COMM':   ('08_Communications', 'GBL_08_Communications'),
}

renamed = []
for key, (old_name, new_name) in renames.items():
    folder_id = ids.get(key)
    if not folder_id:
        print(f'  ⚠️  ID not found for {key}')
        continue
    try:
        drive_service.files().update(
            fileId=folder_id,
            body={'name': new_name},
            supportsAllDrives=True
        ).execute()
        print(f'  ✅ {old_name} → {new_name}')
        renamed.append(new_name)
        time.sleep(0.2)
    except Exception as e:
        print(f'  ❌ Error renaming {old_name}: {e}')

print(f'\n✅ Renamed {len(renamed)}/8 Global Governance folders.')
print('   These now sort after all 99_ Archive folder (GBL_ prefix).')

## STEP 16 — Upload Export Files to Drive

Uploads the 3 registry export files from Colab runtime into `00_Templates/` so they persist permanently in Drive.

In [ ]:
section('STEP 16 — Upload registry export files to 00_Templates')

import os

files_to_upload = [
    ('CPI_BGD_Folder_Registry.csv',              'text/csv'),
    ('CPI_BGD_Mission_OS_Folder_Registry.xlsx',  'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'),
    ('CPI_BGD_OS_Build_Summary.json',            'application/json'),
]

tpl_folder_id = ids['00_TPL']
uploaded = []

for filename, mime in files_to_upload:
    if not os.path.exists(filename):
        print(f'  ⚠️  File not found in Colab runtime: {filename}')
        print(f'      → Download from Colab Files panel and re-upload manually.')
        continue

    # Check if already uploaded
    existing = find_item(filename, tpl_folder_id)
    if existing:
        print(f'  📎 [EXISTS]  {filename}')
        continue

    meta  = {'name': filename, 'parents': [tpl_folder_id]}
    media = MediaIoBaseUpload(open(filename, 'rb'), mimetype=mime)
    f = drive_service.files().create(
        body=meta, media_body=media, fields='id', supportsAllDrives=True
    ).execute()
    print(f'  📎 [UPLOADED] {filename} → {f["id"]}')
    uploaded.append(filename)
    time.sleep(0.3)

print(f'\n✅ Uploaded {len(uploaded)} file(s) to 00_Templates in Drive.')

## STEP 17 — Populate the INDEX Sheet

Writes all 99 folder IDs with full Drive URLs into the INDEX sheet (`1jq5y9rY5-WaTtukok7Sol0px5SM5fUG6tF3MQmvBJaQ`).

In [ ]:
section('STEP 17 — Populate INDEX Sheet with all folder IDs + URLs')

INDEX_SHEET_ID = '1jq5y9rY5-WaTtukok7Sol0px5SM5fUG6tF3MQmvBJaQ'

# Full folder map with human-readable paths
INDEX_ROWS = [
    # [Item_ID, Item_Type, Display_Name, Full_Path, Drive_ID, Layer, Program, Status]
    ['F001','Folder','00_Templates',          '000_BGD_CPMS/00_Templates',                          '1Qkyi-ZdzvG_52-aFp471nzAvTCLMUAzG','Mission','All',   'Active'],
    ['F002','Folder','01_HRA',                '000_BGD_CPMS/01_HRA',                                '18iryBvBOZr96FR49zZoVO6q5QZQX3wdp','Mission','HRA',   'Active'],
    ['F003','Folder','02_G&F',               '000_BGD_CPMS/02_G&F',                                '1zDhsKUZEQi50mQjpdU8TcvmaMPJ2Tf-N','Mission','G&F',   'Active'],
    ['F004','Folder','03_LSC',               '000_BGD_CPMS/03_LSC',                                '1U_Tf3Z-kYAurOTkIZ-BLhrqy3dIDYaHo','Mission','LSC',   'Active'],
    ['F005','Folder','04_MER',               '000_BGD_CPMS/04_MER',                                '1LH5oT8sIb0A_taNNok2HBAZ0C6PWE4Qj','Mission','MER',   'Active'],
    ['F006','Folder','GBL_01_Governance',    '000_BGD_CPMS/GBL_01_Governance',                     '1G-Gv9WDFuEwg8sogFYf8jjag_USjiipk','Global', 'GOV',   'Active'],
    ['F007','Folder','GBL_02_Finance',       '000_BGD_CPMS/GBL_02_Finance',                        '1mNZ18VrLZGVhBdD8_wuuwegJyjXHt4zV','Global', 'G&F',   'Active'],
    ['F008','Folder','GBL_03_HR_Admin',      '000_BGD_CPMS/GBL_03_HR_Admin',                       '1DSQW6C8vV1fuBAqNldmnprKU4x3m2XZQ','Global', 'HRA',   'Active'],
    ['F009','Folder','GBL_04_Logistics',     '000_BGD_CPMS/GBL_04_Logistics',                      '1rwVQDKpGQeSWQNvOKNSWPb5K7M_98vuA','Global', 'LSC',   'Active'],
    ['F010','Folder','GBL_05_Programs',      '000_BGD_CPMS/GBL_05_Programs',                       '1408RoCljObPWIVhGkl58nntK0-9Dd_ex','Global', 'Programs','Active'],
    ['F011','Folder','GBL_06_MER',           '000_BGD_CPMS/GBL_06_MER',                            '1MCzNI8dQTNsoS4fWXg1ib-bKIflOzpW_','Global', 'MER',   'Active'],
    ['F012','Folder','GBL_07_Partnerships',  '000_BGD_CPMS/GBL_07_Partnerships',                   '1-WaJfRHzcozL58xFdYGy8zA2jJLPhMc6','Global', 'G&F',   'Active'],
    ['F013','Folder','GBL_08_Communications','000_BGD_CPMS/GBL_08_Communications',                 '1WTi2Sqtzz45DUvGJV9uWXggZv317wazk','Global', 'MC',    'Active'],
    # Key sheets
    ['S001','Sheet','CPI_BGD_TemplateRegistry_Master_v1.0','00_Templates',                          '1kl39tYBZ2qVq5eq0LRv7e4Cv-jXeNUPGdIrZhrw1mIM','Mission','All','Active'],
    ['S002','Sheet','INDEX',                 '000_BGD_CPMS root',                                  '1jq5y9rY5-WaTtukok7Sol0px5SM5fUG6tF3MQmvBJaQ','Mission','All','Active'],
]

# Build full rows with Drive URLs
def folder_url(fid):
    return f'https://drive.google.com/drive/folders/{fid}'

def sheet_url(sid):
    return f'https://docs.google.com/spreadsheets/d/{sid}'

HEADER = ['Item_ID','Item_Type','Display_Name','Full_Path','Drive_ID',
          'Drive_URL','Layer','Program','Status','Last_Updated']

data_rows = []
for row in INDEX_ROWS:
    fid  = row[4]
    itype= row[1]
    url  = sheet_url(fid) if itype == 'Sheet' else folder_url(fid)
    full = row[:5] + [url] + row[5:] + [TODAY]
    data_rows.append(full)

# Write to INDEX sheet
all_values = [HEADER] + data_rows

sheets_service.spreadsheets().values().clear(
    spreadsheetId=INDEX_SHEET_ID, range='Sheet1'
).execute()

sheets_service.spreadsheets().values().update(
    spreadsheetId=INDEX_SHEET_ID,
    range='Sheet1!A1',
    valueInputOption='RAW',
    body={'values': all_values}
).execute()

print(f'✅ INDEX sheet populated with {len(data_rows)} rows.')
print(f'   Open: https://docs.google.com/spreadsheets/d/{INDEX_SHEET_ID}')

## STEP 18 — Apply CPI Permission Rules

Sets folder-level permissions: `00_Templates` becomes View-only for general staff. `02_G&F` is restricted to finance team.

In [ ]:
section('STEP 18 — Apply folder permission rules')

# Define staff email groups (replace with real emails)
# For Shared Drives, permissions work at file/folder level
# These are example additions — adjust to your actual team

PERMISSION_RULES = [
    # (folder_id_key, email, role)
    # role: 'reader' = view only, 'writer' = editor
    # Admin / CR always has full access via Shared Drive membership
    # Below: restrict or grant specific folder access

    # ariful@cpintl.org already has full access as drive member
    # Add other staff here:
    # ('02_GF', 'finance.manager@cpintl.org', 'writer'),
    # ('01_HRA', 'hr.manager@cpintl.org', 'writer'),
    # ('04_MER', 'mer.officer@cpintl.org', 'writer'),
]

if not PERMISSION_RULES:
    print('⚠️  No permission rules defined yet.')
    print('   Add staff emails to PERMISSION_RULES list and re-run.')
    print("   Example: ('02_GF', 'finance@cpintl.org', 'writer')")
else:
    for folder_key, email, role in PERMISSION_RULES:
        folder_id = ids.get(folder_key)
        if not folder_id:
            print(f'  ⚠️  Folder ID not found: {folder_key}')
            continue
        try:
            drive_service.permissions().create(
                fileId=folder_id,
                body={'type': 'user', 'role': role, 'emailAddress': email},
                supportsAllDrives=True
            ).execute()
            print(f'  ✅ {folder_key}: {role} → {email}')
            time.sleep(0.2)
        except Exception as e:
            print(f'  ❌ Error: {e}')

print('\n✅ Permission setup complete.')

## STEP 19 — Generate Full INDEX with all 99 Folder IDs

Reads the Folder Registry CSV and writes ALL 99 folder rows into the INDEX sheet.

In [ ]:
section('STEP 19 — Full 99-folder INDEX population from CSV')

import csv, os

INDEX_SHEET_ID = '1jq5y9rY5-WaTtukok7Sol0px5SM5fUG6tF3MQmvBJaQ'
CSV_FILE = 'CPI_BGD_Folder_Registry.csv'

if not os.path.exists(CSV_FILE):
    print(f'⚠️  {CSV_FILE} not found. Upload it via Colab Files panel first.')
else:
    with open(CSV_FILE) as f:
        reader = csv.DictReader(f)
        csv_rows = list(reader)

    HEADER = ['Item_ID','Folder_Key','Drive_ID','Drive_URL','Status','Last_Updated']
    data_rows = []
    for i, row in enumerate(csv_rows, 1):
        fid = row.get('Drive_ID', '')
        url = f'https://drive.google.com/drive/folders/{fid}' if fid else ''
        data_rows.append([
            f'F{i:03d}',
            row.get('Folder_Key', ''),
            fid,
            url,
            row.get('Status', 'Verified'),
            TODAY
        ])

    sheets_service.spreadsheets().values().update(
        spreadsheetId=INDEX_SHEET_ID,
        range='Sheet2!A1',
        valueInputOption='RAW',
        body={'values': [HEADER] + data_rows}
    ).execute()

    print(f'✅ All {len(data_rows)} folder rows written to INDEX Sheet2.')
    print(f'   Open: https://docs.google.com/spreadsheets/d/{INDEX_SHEET_ID}')

---
## Summary of Phase 2
| Step | Action | Result |
|---|---|---|
| 15 | Rename Global folders | `GBL_` prefix — no naming conflict |
| 16 | Upload exports to Drive | CSV/Excel/JSON permanently in 00_Templates |
| 17 | Populate INDEX sheet | 16 key folders + sheets with live URLs |
| 18 | Permission rules | Framework ready — add staff emails |
| 19 | Full 99-folder INDEX | All folders indexed in Sheet2 |